<a href="https://colab.research.google.com/github/EthanIsaac/valorant-voiceline-qa/blob/main/notebooks/02_speech_to_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Speech-to-Text: comparing 3 pre-trained models

**Project:** VALORANT Voice-Line QA & Localization Assistant (Riot Games)
**Pipeline:** `automatic-speech-recognition`, which transcribes a voice line so it can be **checked against the script**, and becomes the source text for the Spanish subtitles (notebook 03).

**Why it matters:** a recording that does not say what the script says (wrong take, cut-off word, mislabelled file) is a bug. Checking thousands of lines by ear is slow; a transcript lets the QA team review only the lines that do not match.

**Notebook outline**
- Step 0: Setup (packages, seed, output folder)
- Step 1: Load the test clips (same split as notebook 01)
- Step 2: Choose the 3 candidate models and how to score them
- Step 3: Transcribe the test set and score each model (WER and exact match)
- Step 4: Measure CPU runtime through a `pipeline` (Streamlit Cloud runs on CPU)
- Step 5: Compare the models, pick the best and look at its errors
- Step 6: Improvement attempt: a VALORANT vocabulary prompt for the selected model (no training)
- Save and download the results (with SHA-256 fingerprints)

All results are written to `results/speech_to_text/` so the report can quote them exactly.
No model is fine-tuned here: the models are used as published.

> Runtime: **GPU** (Runtime → Change runtime type → T4 GPU). The GPU only speeds up transcription (Steps 3 and 6); runtime is measured on CPU.

# Step 0: Setup

In [ ]:
# Same transformers constraint as the course notebooks; jiwer computes the word error rate.
# Expected: a pip "dependency conflicts" message about diffusers/gradio needing a newer
# huggingface-hub. Those packages are not used here, so it can be ignored.
%pip install -q "transformers>=4.46.0,<5.0.0" accelerate jiwer

In [ ]:
import hashlib
import importlib.metadata
import io
import json
import os
import time
import urllib.request

import jiwer
import librosa
import numpy as np
import pandas as pd
import soundfile as sf
import torch
import transformers
from IPython.display import Audio, display
from transformers import set_seed

# One seed for everything; transcription itself is deterministic (greedy decoding, no sampling)
SEED = 42
set_seed(SEED)

# Every result file produced by this notebook goes in this folder
RESULTS_DIR = "results/speech_to_text"
os.makedirs(RESULTS_DIR, exist_ok=True)

JIWER_VERSION = importlib.metadata.version("jiwer")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"transformers {transformers.__version__} | torch {torch.__version__} | jiwer {JIWER_VERSION} | device: {device}")
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Step 1: Load the test clips

We reuse the **exact split from notebook 01** (`results/agent_classifier/splits.csv`), downloaded from the project's GitHub repository at a fixed commit and verified with its SHA-256 fingerprint.
This keeps the three pipelines evaluated on the same unseen test clips.

Only clips flagged **`english_only`** are used: they contain speech, and no foreign words or translator notes, so the reference text is exactly what is said in English.
The reference is the cleaned `spoken_text` column (the quoted words of the wiki transcript).

The **validation** clips are loaded too: they are only used in Step 6, to decide on an improvement without touching the test set.

The audio comes from the same dataset revision as notebook 01 (about 2 GB download).

In [ ]:
REPO_COMMIT = "2837dcb342749a346d4370e63545c575ba5c2a07"   # commit with the notebook 01 results
SPLITS_URL = f"https://raw.githubusercontent.com/EthanIsaac/valorant-voiceline-qa/{REPO_COMMIT}/results/agent_classifier/splits.csv"
SPLITS_SHA256 = "72b658b97258de7bc3e77a2b88908080dd46be76d0162c8905cceff2c903dd7d"   # from results/agent_classifier/SHA256SUMS

splits_bytes = urllib.request.urlopen(SPLITS_URL).read()
assert hashlib.sha256(splits_bytes).hexdigest() == SPLITS_SHA256, "splits.csv does not match notebook 01"
splits = pd.read_csv(io.BytesIO(splits_bytes))

clips = splits[splits["split"].isin(["test", "validation"]) & splits["english_only"]].reset_index(drop=True)
test = clips[clips["split"] == "test"].reset_index(drop=True)
print(f"{(clips['split'] == 'validation').sum()} English validation clips (used in Step 6 only)")
print(f"{len(test)} English test clips from {test['agent'].nunique()} agents "
      f"(of {(splits['split'] == 'test').sum()} test clips)")
print(f"Words per line: median {test['spoken_text'].str.split().str.len().median():.0f}, "
      f"max {test['spoken_text'].str.split().str.len().max()} | "
      f"duration: median {test['duration_s'].median():.2f}s, max {test['duration_s'].max():.2f}s")

In [ ]:
from huggingface_hub import hf_hub_download, list_repo_files

DATASET = "NgThVinh/ValorantAgentVoiceLines"
REVISION = "c2e0d3493f44da5be07090d6ebb9b69ebbcb2060"   # same pinned commit as notebook 01
TARGET_SR = 16_000   # all three models expect 16 kHz audio

parquet_files = sorted(
    f for f in list_repo_files(DATASET, repo_type="dataset", revision=REVISION) if f.endswith(".parquet")
)
audio_bytes = {}
for file in parquet_files:
    agent = file.split("/")[0]
    agent_df = pd.read_parquet(hf_hub_download(DATASET, file, repo_type="dataset", revision=REVISION))
    wanted = set(clips.loc[clips["agent"] == agent, "audio_name"])
    for name, audio in zip(agent_df["audio_name"], agent_df["audio_file"]):
        if name in wanted:
            audio_bytes[(agent, name)] = audio["bytes"]


def load_waveform(wav_bytes):
    """WAV bytes -> mono float32 waveform at 16 kHz. Not trimmed: every word must be heard."""
    waveform, sr = sf.read(io.BytesIO(wav_bytes), dtype="float32")
    if waveform.ndim > 1:
        waveform = waveform.mean(axis=1)
    return librosa.resample(waveform, orig_sr=sr, target_sr=TARGET_SR)


clips["waveform"] = [load_waveform(audio_bytes[(a, n)]) for a, n in zip(clips["agent"], clips["audio_name"])]
del audio_bytes
test = clips[clips["split"] == "test"].reset_index(drop=True)
validation = clips[clips["split"] == "validation"].reset_index(drop=True)
print(f"Loaded {len(test)} test and {len(validation)} validation clips")

# Listen to a few examples
for _, row in test.sample(3, random_state=SEED).iterrows():
    print(f"{row['agent']:<10} \"{row['spoken_text']}\"")
    display(Audio(row["waveform"], rate=TARGET_SR))

# Step 2: Choose the 3 candidate models and how to score them

| Key | Hugging Face model | Size | Why it's a candidate |
|---|---|---|---|
| `wav2vec2` | `facebook/wav2vec2-base-960h` | ~95M | Classic speech recogniser (CTC): predicts letters frame by frame, fast and simple. Same family as the agent classifier |
| `whisper_tiny` | `openai/whisper-tiny.en` | ~39M | Smallest Whisper: an encoder-decoder trained on 680k hours of web audio, outputs punctuated text |
| `whisper_base` | `openai/whisper-base.en` | ~74M | Next Whisper size: shows what a larger model buys in accuracy, and costs in runtime |

**Why these three:** they compare two approaches (letter-by-letter CTC vs. a Whisper decoder that writes whole sentences) and two Whisper sizes, all small enough for the CPU-only Streamlit Cloud.
The **English-only** (`.en`) Whisper versions are used because the source voice lines are English; they are more accurate on English than the multilingual versions of the same size.
*Not chosen:* larger Whisper models (small and above, 244M+ parameters) are too slow on a CPU for an interactive app.

**Scoring**
- **Word error rate (WER)**: (substituted + deleted + inserted words) / words in the reference, over the whole test set. Lower is better; 0 is perfect.
- **Exact match**: share of clips whose transcript equals the script line. For the QA team this is the share of lines that **pass automatically**; every other line would go to a human for review.
- **Text normalisation before scoring:** models differ in style (wav2vec2 writes `LETS GO A` in capitals without punctuation; Whisper writes `Let's go A!`). Both the reference and the transcript go through Whisper's English normaliser (lower case, no punctuation, contractions expanded, numbers as digits), which is the standard for comparing speech recognisers, so only **wrong words** count as errors, not style.

In [ ]:
from transformers import AutoTokenizer, pipeline
from transformers.models.whisper.english_normalizer import EnglishTextNormalizer

CANDIDATES = {
    "wav2vec2": "facebook/wav2vec2-base-960h",
    "whisper_tiny": "openai/whisper-tiny.en",
    "whisper_base": "openai/whisper-base.en",
}

normalizer = EnglishTextNormalizer(AutoTokenizer.from_pretrained("openai/whisper-base.en").english_spelling_normalizer)

# The same normaliser is applied to the references and to every model's output
test["reference"] = test["spoken_text"].map(normalizer)
assert (test["reference"] != "").all()

for raw in ["LETS GO A", "Let's go A!", "She's still gunning for ya!", "Two enemies, B site."]:
    print(f"{raw!r:32} -> {normalizer(raw)!r}")

# Step 3: Transcribe the test set and score each model

Each model transcribes every test clip through the same `pipeline("automatic-speech-recognition")` the app uses, **one clip at a time** (like the app; it also avoids padding, which would change the output of wav2vec2-base-960h because it uses no attention mask).
Decoding is greedy (no sampling), so the transcripts are the same on every run.
The GPU is used here only to make this step faster.

In [ ]:
transcripts = test[["agent", "audio_name", "duration_s", "spoken_text", "reference"]].copy()
results = {}

for key, model_id in CANDIDATES.items():
    asr = pipeline("automatic-speech-recognition", model=model_id, device=0 if device == "cuda" else -1)
    start = time.perf_counter()
    raw = [asr({"raw": w, "sampling_rate": TARGET_SR})["text"].strip() for w in test["waveform"]]
    elapsed = time.perf_counter() - start

    hypothesis = [normalizer(t) for t in raw]
    transcripts[f"{key}_raw"] = raw
    transcripts[f"{key}_normalized"] = hypothesis
    transcripts[f"{key}_clip_wer"] = [jiwer.wer(r, h) for r, h in zip(transcripts["reference"], hypothesis)]

    measures = jiwer.process_words(list(transcripts["reference"]), hypothesis)
    results[key] = {
        "model_id": model_id,
        "params_millions": round(asr.model.num_parameters() / 1e6, 1),
        "wer": measures.wer,
        "exact_match": float(np.mean(transcripts[f"{key}_clip_wer"] == 0)),
        "substitutions": measures.substitutions,
        "deletions": measures.deletions,
        "insertions": measures.insertions,
        "reference_words": measures.hits + measures.substitutions + measures.deletions,
        "gpu_seconds_test_set": round(elapsed, 1),
    }
    print(f"{key:<13} WER {measures.wer:.3f} | exact match {results[key]['exact_match']:.1%} | {elapsed:.0f}s for {len(test)} clips")
    del asr
    torch.cuda.empty_cache()

transcripts.to_csv(f"{RESULTS_DIR}/test_transcripts.csv", index=False)
pd.DataFrame(results).T

In [ ]:
# The same lines side by side
cols = ["spoken_text"] + [f"{key}_raw" for key in CANDIDATES]
transcripts[cols].sample(10, random_state=SEED)

# Step 4: Measure CPU runtime through a `pipeline`

The deployed app runs on **Streamlit Cloud, which has no GPU**, so runtime is measured on CPU with the same `pipeline` call the app will use.

Following the course method, runtime is timed both:
- **with model loading**: `cpu_load_s`, the one-off cost when the app starts (the model files are downloaded first, so network speed is not counted)
- **without model loading**: `cpu_ms_per_clip`, the cost of each transcription, over the first 100 test clips

As a consistency check, the CPU transcripts must match the Step 3 transcripts (`cpu_agreement`).

In [ ]:
from huggingface_hub import snapshot_download

speed_clips = test.head(100)

for key, model_id in CANDIDATES.items():
    snapshot_download(model_id)   # make sure the files are cached, so only loading is timed

    # With model loading
    start = time.perf_counter()
    asr = pipeline("automatic-speech-recognition", model=model_id, device=-1)   # device=-1 means CPU
    load_s = time.perf_counter() - start

    asr({"raw": speed_clips["waveform"].iloc[0], "sampling_rate": TARGET_SR})   # warm-up

    # Without model loading: one clip at a time, like the app
    start = time.perf_counter()
    raw = [asr({"raw": w, "sampling_rate": TARGET_SR})["text"].strip() for w in speed_clips["waveform"]]
    ms_per_clip = (time.perf_counter() - start) / len(speed_clips) * 1000

    agreement = np.mean([normalizer(t) == h for t, h in zip(raw, transcripts[f"{key}_normalized"].head(100))])
    results[key].update({
        "cpu_load_s": round(load_s, 2),
        "cpu_ms_per_clip": round(ms_per_clip, 1),
        "cpu_agreement": agreement,
    })
    print(f"{key:<13} load={load_s:5.2f}s  per clip={ms_per_clip:6.1f}ms  agreement with Step 3={agreement:.0%}")
    del asr

# Step 5: Compare the models and pick the best

**Selection rule** (decided before running):
1. Lowest **WER** on the test set.
2. If several models are within 0.01 of the best WER, choose the **fastest on CPU**, because the app runs on CPU.

No model is trained here and the three models are used as published, so the test set only measures each one once. (Step 6 does use the validation set, because it makes a design choice.)

In [ ]:
comparison = pd.DataFrame(results).T.infer_objects().rename_axis("model").reset_index()

best_wer = comparison["wer"].min()
close = comparison[comparison["wer"] <= best_wer + 0.01]
BEST = close.sort_values("cpu_ms_per_clip").iloc[0]["model"]
comparison["selected"] = comparison["model"] == BEST

comparison.to_csv(f"{RESULTS_DIR}/model_comparison.csv", index=False)
print("Selected:", BEST, "->", CANDIDATES[BEST])
comparison

In [ ]:
# Where does the selected model go wrong? Most frequent word errors over the test set
def top_word_errors(references, hypotheses, n=20):
    """The n most frequent substituted, deleted and inserted words."""
    substitutions, insertions, deletions = jiwer.collect_error_counts(
        jiwer.process_words(list(references), list(hypotheses))
    )
    errors = pd.DataFrame(
        [("substitution", ref, hyp, c) for (ref, hyp), c in substitutions.items()]
        + [("deletion", word, "", c) for word, c in deletions.items()]
        + [("insertion", "", word, c) for word, c in insertions.items()],
        columns=["error", "script word", "transcribed as", "count"],
    )
    errors = errors.sort_values(["count", "error", "script word", "transcribed as"], ascending=[False, True, True, True])
    return errors.head(n).reset_index(drop=True)


errors = top_word_errors(transcripts["reference"], transcripts[f"{BEST}_normalized"])
errors.to_csv(f"{RESULTS_DIR}/{BEST}_top_word_errors.csv", index=False)
errors

In [ ]:
# Lines the selected model got most wrong (these would go to a human reviewer)
worst = transcripts.sort_values([f"{BEST}_clip_wer", "agent", "audio_name"], ascending=[False, True, True])
worst[["agent", "spoken_text", f"{BEST}_raw", f"{BEST}_clip_wer"]].head(10)

# Step 6: Improvement attempt: a VALORANT vocabulary prompt

**What Step 5 shows.** Most of the selected model's errors are **game vocabulary**, not general speech:
site callouts ("C" heard as *see* / *sea*, "B" as *be*), game terms (*spike*, *Mid*) and agent names (*Jett* → *jet*, *Reyna* → *Rayna*).
A general-purpose model has no reason to expect these words.

**The idea (no training).** Whisper accepts a short text **prompt**, which it treats as the text spoken just before the clip.
Giving it a glossary of VALORANT callouts and agent names makes those spellings more likely. The QA team could maintain this glossary as the game grows.
The risk: on very short clips the prompt can push the model to *produce* words that were not said, or get stuck repeating itself.
To see this, the number of **runaway transcripts** (at least 3× longer than the script line) is counted for both variants.

**How it is evaluated (rule fixed before running):**
1. On the **validation** clips, compare the selected model with and without the prompt.
2. The prompt is **adopted only if it lowers the validation WER**.
3. The test set is used once, at the end, to report the final score of the adopted setup (and the prompt's effect on it).

The prompt is written once below and not changed after seeing results.

In [ ]:
assert BEST.startswith("whisper"), "prompting only applies to Whisper models"

AGENT_NAMES = ["Astra", "Breach", "Brimstone", "Chamber", "Clove", "Cypher", "Deadlock", "Fade", "Gekko",
               "Harbor", "Iso", "Jett", "KAY/O", "Killjoy", "Neon", "Omen", "Phoenix", "Raze", "Reyna",
               "Sage", "Skye", "Sova", "Tejo", "Viper", "Vyse", "Yoru"]
PROMPT = ("VALORANT callouts: A site, B site, C site, Mid, spike, plant, defuse, attackers, defenders. "
          "Agents: " + ", ".join(AGENT_NAMES) + ".")
print(PROMPT)

best_asr = pipeline("automatic-speech-recognition", model=CANDIDATES[BEST], device=0 if device == "cuda" else -1)
prompt_ids = best_asr.tokenizer.get_prompt_ids(PROMPT, return_tensors="pt").to(best_asr.device)


def transcribe(asr, waveforms, prompt_ids=None):
    """One clip at a time, like the app; the prompt is optional."""
    kwargs = {"generate_kwargs": {"prompt_ids": prompt_ids}} if prompt_ids is not None else {}
    return [asr({"raw": w, "sampling_rate": TARGET_SR}, **kwargs)["text"].strip() for w in waveforms]


def runaway_clips(references, hypotheses):
    """Transcripts at least 3x longer than the script line: the model got stuck repeating itself."""
    return int(sum(len(h.split()) >= 3 * len(r.split()) + 3 for r, h in zip(references, hypotheses)))


def score(references, raw_transcripts):
    hypotheses = [normalizer(t) for t in raw_transcripts]
    clip_wer = np.array([jiwer.wer(r, h) for r, h in zip(references, hypotheses)])
    scores = {
        "wer": jiwer.wer(list(references), hypotheses),
        "exact_match": float(np.mean(clip_wer == 0)),
        "runaway_clips": runaway_clips(references, hypotheses),
    }
    return scores, hypotheses, clip_wer

In [ ]:
# 1-2. Decide on the validation set
validation["reference"] = validation["spoken_text"].map(normalizer)
val_scores = {}
for variant, ids in [("no_prompt", None), ("prompt", prompt_ids)]:
    val_scores[variant], _, _ = score(validation["reference"], transcribe(best_asr, validation["waveform"], ids))
    print(f"validation  {variant:<10} WER {val_scores[variant]['wer']:.3f} | exact match {val_scores[variant]['exact_match']:.1%} "
          f"| runaway transcripts {val_scores[variant]['runaway_clips']}")

USE_PROMPT = val_scores["prompt"]["wer"] < val_scores["no_prompt"]["wer"]
print("\nAdopt the prompt:", USE_PROMPT)

In [ ]:
# 3. Report on the test set (the no-prompt test transcripts come from Step 3)
test_raw_prompt = transcribe(best_asr, test["waveform"], prompt_ids)
test_scores = {"no_prompt": {
    "wer": results[BEST]["wer"],
    "exact_match": results[BEST]["exact_match"],
    "runaway_clips": runaway_clips(transcripts["reference"], transcripts[f"{BEST}_normalized"]),
}}
test_scores["prompt"], prompt_normalized, prompt_clip_wer = score(transcripts["reference"], test_raw_prompt)

prompt_transcripts = transcripts[["agent", "audio_name", "spoken_text", "reference",
                                  f"{BEST}_raw", f"{BEST}_clip_wer"]].copy()
prompt_transcripts[f"{BEST}_prompt_raw"] = test_raw_prompt
prompt_transcripts[f"{BEST}_prompt_clip_wer"] = prompt_clip_wer
prompt_transcripts.to_csv(f"{RESULTS_DIR}/{BEST}_prompt_test_transcripts.csv", index=False)

# Effect on the lines the prompt targets: lines that mention a site letter (A, B or C)
mentions_site = transcripts["spoken_text"].str.contains(r"\b[ABC]\b", regex=True).to_numpy()

# CPU runtime with the prompt (same 100 clips and method as Step 4)
cpu_asr = pipeline("automatic-speech-recognition", model=CANDIDATES[BEST], device=-1)
cpu_prompt_ids = cpu_asr.tokenizer.get_prompt_ids(PROMPT, return_tensors="pt")
transcribe(cpu_asr, speed_clips["waveform"].head(1), cpu_prompt_ids)   # warm-up
start = time.perf_counter()
transcribe(cpu_asr, speed_clips["waveform"], cpu_prompt_ids)
prompt_ms_per_clip = (time.perf_counter() - start) / len(speed_clips) * 1000
del cpu_asr

rows = []
for variant, clip_wer in [("no_prompt", transcripts[f"{BEST}_clip_wer"].to_numpy()), ("prompt", prompt_clip_wer)]:
    rows.append({
        "model": CANDIDATES[BEST],
        "variant": variant,
        "val_wer": val_scores[variant]["wer"],
        "val_exact_match": val_scores[variant]["exact_match"],
        "val_runaway_clips": val_scores[variant]["runaway_clips"],
        "test_wer": test_scores[variant]["wer"],
        "test_exact_match": test_scores[variant]["exact_match"],
        "test_runaway_clips": test_scores[variant]["runaway_clips"],
        "test_exact_match_site_lines": float(np.mean(clip_wer[mentions_site] == 0)),
        "test_exact_match_other_lines": float(np.mean(clip_wer[~mentions_site] == 0)),
        "cpu_ms_per_clip": results[BEST]["cpu_ms_per_clip"] if variant == "no_prompt" else round(prompt_ms_per_clip, 1),
        "adopted": (variant == "prompt") == USE_PROMPT,
    })
prompt_experiment = pd.DataFrame(rows)
prompt_experiment.to_csv(f"{RESULTS_DIR}/prompt_experiment.csv", index=False)
print(f"Test lines mentioning a site: {mentions_site.sum()} of {len(test)}")
prompt_experiment

In [ ]:
# Lines the prompt fixed, and lines it broke (test set)
changed = prompt_transcripts[prompt_transcripts[f"{BEST}_clip_wer"] != prompt_transcripts[f"{BEST}_prompt_clip_wer"]]
fixed = changed[changed[f"{BEST}_prompt_clip_wer"] < changed[f"{BEST}_clip_wer"]]
broken = changed[changed[f"{BEST}_prompt_clip_wer"] > changed[f"{BEST}_clip_wer"]]
print(f"Lines improved by the prompt: {len(fixed)} | made worse: {len(broken)}")
cols = ["spoken_text", f"{BEST}_raw", f"{BEST}_prompt_raw"]
display(fixed[cols].head(8))
display(broken[cols].head(8))

In [ ]:
# Most frequent word errors that remain with the prompt
prompt_errors = top_word_errors(transcripts["reference"], prompt_normalized)
prompt_errors.to_csv(f"{RESULTS_DIR}/{BEST}_prompt_top_word_errors.csv", index=False)
prompt_errors

In [ ]:
# Record exactly how these results were produced
with open(f"{RESULTS_DIR}/run_config.json", "w") as f:
    json.dump({
        "seed": SEED,
        "dataset": DATASET,
        "dataset_revision": REVISION,
        "splits_csv": {"url": SPLITS_URL, "sha256": SPLITS_SHA256},
        "test_clips": len(test),
        "validation_clips": len(validation),
        "candidates": CANDIDATES,
        "audio": {"sampling_rate": TARGET_SR, "trimmed": False},
        "decoding": "greedy, one clip at a time",
        "text_normalizer": "transformers EnglishTextNormalizer (Whisper)",
        "selection_rule": "lowest test WER; ties within 0.01 go to lowest cpu_ms_per_clip",
        "selected_model": BEST,
        "prompt_experiment": {
            "prompt": PROMPT,
            "rule": "adopt the prompt only if it lowers the validation WER",
            "adopted": bool(USE_PROMPT),
        },
        "versions": {"transformers": transformers.__version__, "torch": torch.__version__, "jiwer": JIWER_VERSION},
        "gpu": torch.cuda.get_device_name(0) if device == "cuda" else None,
        "run_at_utc": pd.Timestamp.now(tz="UTC").isoformat(timespec="seconds"),
    }, f, indent=2)
print(open(f"{RESULTS_DIR}/run_config.json").read())

# Save and download the results

Every file this notebook generated is listed with its **SHA-256 fingerprint** in `SHA256SUMS`.
The transcripts should be identical on a rerun (greedy decoding); the runtime files differ slightly because they contain timings.
Then everything is zipped and downloaded as `results_speech_to_text.zip`.

In [ ]:
from pathlib import Path

result_files = sorted(p for p in Path(RESULTS_DIR).rglob("*") if p.is_file() and p.name != "SHA256SUMS")
with open(f"{RESULTS_DIR}/SHA256SUMS", "w") as f:
    for p in result_files:
        digest = hashlib.sha256(p.read_bytes()).hexdigest()
        f.write(f"{digest}  {p.relative_to(RESULTS_DIR)}\n")
        print(f"{digest[:16]}…  {p.relative_to(RESULTS_DIR)}  ({p.stat().st_size:,} bytes)")

In [ ]:
!rm -f results_speech_to_text.zip && zip -qrX results_speech_to_text.zip results/speech_to_text
from google.colab import files
files.download("results_speech_to_text.zip")